# 05 — LDP (Lakeflow Declarative Pipelines) Expectations: The Lakehouse Answer to Engine-Level Constraints

In the RDBMS world, you had a layered defense:
- **CHECK constraints** caught basic data quality issues (nulls, format)
- **PRIMARY KEY / UNIQUE** prevented duplicates at the storage level
- **FOREIGN KEY** enforced referential integrity
- **Stored procedures & triggers** added custom business logic

On the lakehouse, **LDP (Lakeflow Declarative Pipelines)** expectations replace all of that in a single,
declarative framework. You define your rules once in the pipeline, and LDP enforces them
on every run — no matter who or what upstream is writing data.

### Three enforcement modes (think: WARNING → FILTER → HARD STOP)

| LDP Mode | RDBMS Equivalent | Behavior |
|----------|------------------|----------|
| `expect` | A CHECK constraint that logs but doesn't reject | **Warn**: record the violation as a metric, keep the row |
| `expect_or_drop` | A BEFORE INSERT trigger that silently filters | **Drop**: remove violating rows, pipeline continues |
| `expect_or_fail` | A PRIMARY KEY / UNIQUE constraint that rejects the write | **Fail**: halt the entire pipeline, no bad data gets through |

### Where to enforce what (Bronze → Silver → Gold)

| Layer | Enforcement Level | Why |
|-------|-------------------|-----|
| **Bronze** | None — accept everything | Raw landing zone. You *want* duplicates here so you can debug what upstream sent you. Deleting them at ingest loses forensic visibility. |
| **Silver** | `expect_or_drop` — filter bad rows | Clean and conform. Drop nulls, malformed emails, obviously invalid records. Deduplicate with window functions. This is your staging area. |
| **Gold** | `expect_or_fail` — hard stop on violations | Business-ready data. If a duplicate PK makes it to Gold, something is fundamentally broken upstream and the pipeline should **stop**, not silently corrupt reporting. This is your PK/UNIQUE equivalent. |

This notebook:
1. Uploads `05_dlt_pipeline_code.py` to the workspace
2. Creates an LDP pipeline via the SDK
3. Runs it with clean data → pipeline **succeeds**
4. Runs it with duplicate data → pipeline **fails** on `expect_or_fail` (just like a PK violation would)
5. Cleans up

In [1]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.pipelines import (
    PipelineLibrary,
    NotebookLibrary,
    PipelineCluster,
)
from dotenv import load_dotenv
import os
import time

load_dotenv()

w = WorkspaceClient()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "pk_unique_demo")

WORKSPACE_PATH = f"/Users/{w.current_user.me().user_name}/pk_unique_demo"
PIPELINE_NAME = "pk_unique_demo_dlt"

print(f"Workspace path: {WORKSPACE_PATH}")
print(f"Pipeline name: {PIPELINE_NAME}")

Workspace path: /Users/alexander.booth@databricks.com/pk_unique_demo
Pipeline name: pk_unique_demo_dlt


## Step 1: Upload pipeline code to workspace

In [2]:
import base64

# Read the local pipeline code
with open("05_dlt_pipeline_code.py", "r") as f:
    pipeline_code = f.read()

# Ensure the workspace directory exists
w.workspace.mkdirs(WORKSPACE_PATH)

# Upload the pipeline code
remote_path = f"{WORKSPACE_PATH}/05_dlt_pipeline_code.py"
w.workspace.upload(
    path=remote_path,
    content=pipeline_code.encode("utf-8"),
    overwrite=True,
)

print(f"Uploaded pipeline code to: {remote_path}")

Uploaded pipeline code to: /Users/alexander.booth@databricks.com/pk_unique_demo/05_dlt_pipeline_code.py


## Step 2: Create the LDP pipeline

In [3]:
pipeline = w.pipelines.create(
    name=PIPELINE_NAME,
    serverless=True,
    channel="CURRENT",
    catalog=UC_CATALOG,
    target=f"{UC_SCHEMA}_dlt",
    development=True,  # No update-level retries on failure
    libraries=[
        PipelineLibrary(
            notebook=NotebookLibrary(path=remote_path)
        )
    ],
    configuration={
        "uc_catalog": UC_CATALOG,
        "uc_schema": UC_SCHEMA,
        "pipelines.maxFlowRetryAttempts": "0",  # Fail once, don't retry the flow
    },
    continuous=False,
)

pipeline_id = pipeline.pipeline_id
print(f"Created pipeline: {pipeline_id}")

Created pipeline: 7226a463-f784-4a2d-a27a-0a2bd5d73fcf


## Step 3: Prepare clean data and run the pipeline

First run: we'll feed in only the 10 genuinely new customers (no duplicates).
Think of this as a normal day — data arrives, pipeline validates it, Gold layer is clean.

This is equivalent to your Oracle stored procedure accepting a batch of valid INSERTs.

In [4]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.serverless().getOrCreate()

# Save the full batch for later, then overwrite with clean-only data
spark.sql(f"""
    CREATE OR REPLACE TABLE {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch_full AS
    SELECT * FROM {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch
""")

# Keep only genuinely new customers (customer_id > 200, unique emails)
spark.sql(f"""
    CREATE OR REPLACE TABLE {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch AS
    SELECT * FROM {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch_full
    WHERE customer_id > 200
    AND email NOT IN (SELECT email FROM {UC_CATALOG}.{UC_SCHEMA}.customers)
""")

clean_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch").collect()[0].cnt
print(f"Clean batch: {clean_count} rows (no duplicates)")

Clean batch: 0 rows (no duplicates)


In [5]:
def run_pipeline_and_wait(pipeline_id, description=""):
    """Trigger a pipeline update and poll until completion."""
    print(f"Starting pipeline update{f' ({description})' if description else ''}...")
    update = w.pipelines.start_update(pipeline_id=pipeline_id, full_refresh=True)
    update_id = update.update_id
    print(f"Update ID: {update_id}")

    while True:
        status = w.pipelines.get(pipeline_id=pipeline_id)
        state = status.latest_updates[0].state.value if status.latest_updates else "UNKNOWN"
        print(f"  State: {state}")
        if state in ("COMPLETED", "FAILED", "CANCELED"):
            break
        time.sleep(15)

    return state


# Run with clean data
state = run_pipeline_and_wait(pipeline_id, "clean data — should succeed")
print(f"\nResult: {state}")

Starting pipeline update (clean data — should succeed)...
Update ID: de0ead3e-f6a8-4e06-81f3-da7ce169c708
  State: CREATED
  State: WAITING_FOR_RESOURCES
  State: INITIALIZING
  State: RUNNING
  State: COMPLETED

Result: COMPLETED


## Step 4: Swap in dirty data and run again

We restore the full batch and **add intra-batch duplicates** (same `customer_id`/`email` repeated within the batch).
The Gold `expect_or_fail` checks for duplicates *within* the batch — notebook 01's "duplicates" are vs the main
customers table, so each batch row was unique; we must add repeated rows to trigger the failure.

In Oracle, you'd get `ORA-00001: unique constraint violated`. Here, the LDP pipeline **FAILs** at the Gold layer
on `expect_or_fail` — same outcome, different mechanism.

In [6]:
# Restore the full batch and add INTRA-BATCH duplicates.
# The Gold expect_or_fail checks duplicates WITHIN the batch (id_count=1, email_count=1).
# Notebook 01's "duplicates" are vs the main customers table — each batch row is unique within the batch.
# So we must add rows that repeat customer_id/email WITHIN this batch to trigger expect_or_fail.
# Duplicate one row twice so we have 3 rows with same customer_id & email within the batch
dup_row = f"SELECT * FROM {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch_full WHERE customer_id = (SELECT MIN(customer_id) FROM {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch_full)"
spark.sql(f"""
    CREATE OR REPLACE TABLE {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch AS
    SELECT * FROM {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch_full
    UNION ALL ({dup_row})
    UNION ALL ({dup_row})
""")

dirty_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch").collect()[0].cnt
dup_check = spark.sql(f"""
    SELECT customer_id, COUNT(*) AS n FROM {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch
    GROUP BY customer_id HAVING COUNT(*) > 1
""")
print(f"Dirty batch: {dirty_count} rows (includes intra-batch duplicates)")
if dup_check.count() > 0:
    print("  Duplicate customer_ids within batch:", [r.customer_id for r in dup_check.collect()])

# Run again — this should fail
state = run_pipeline_and_wait(pipeline_id, "dirty data — should FAIL")
print(f"\nResult: {state}")

Dirty batch: 38 rows (includes intra-batch duplicates)
  Duplicate customer_ids within batch: [8]
Starting pipeline update (dirty data — should FAIL)...
Update ID: 4a0704b5-1f59-45e5-bd1c-981d0cb85025
  State: CREATED
  State: RUNNING
  State: FAILED

Result: FAILED


## Step 5: Inspect the pipeline event log

Expectation metrics (passed/failed/dropped) are in the event log. **Note:** `expect_or_drop` does *not* create a quarantine table — dropped rows are discarded; only counts appear in the event log. To retain failures, you'd need a custom pattern (e.g. `expect` + a separate capture flow).

In [7]:
# Expectation failure details live in the event log, not the REST API.
# Query event_log(pipeline_id) — only the pipeline owner can query it.
try:
    df = spark.sql(f"""
        WITH latest_update AS (
            SELECT origin.update_id AS id
            FROM event_log('{pipeline_id}')
            WHERE event_type = 'create_update'
            ORDER BY timestamp DESC
            LIMIT 1
        ),
        expectations_parsed AS (
            SELECT
                origin.flow_name AS table_name,
                origin.update_id,
                explode(
                    from_json(
                        details:flow_progress.data_quality.expectations,
                        "array<struct<name: string, dataset: string, passed_records: int, failed_records: int>>"
                    )
                ) row_exp
            FROM event_log('{pipeline_id}')
            CROSS JOIN latest_update
            WHERE event_type = 'flow_progress'
              AND origin.update_id = latest_update.id
              AND details:flow_progress.data_quality.expectations IS NOT NULL
        )
        SELECT
            table_name,
            row_exp.name AS expectation,
            row_exp.passed_records,
            row_exp.failed_records
        FROM expectations_parsed
        ORDER BY table_name, expectation
    """)
    print("Expectation metrics (from event log):")
    print("=" * 80)
    df.show(truncate=False)
    # Highlight failures
    failures = df.filter("failed_records > 0")
    if failures.count() > 0:
        print("\n>>> EXPECTATION FAILURES (expect_or_fail triggered):")
        failures.show(truncate=False)
except Exception as e:
    print(f"Could not query event log (pipeline owner only): {e}")
    print("Fallback — REST API events:")
    for event in list(w.pipelines.list_pipeline_events(pipeline_id=pipeline_id, max_results=20)):
        if event.event_type in ("flow_progress", "update_progress") and event.message:
            print(f"  [{event.event_type}] {event.message}")

Expectation metrics (from event log):
+---------------------------------------------------+--------------------+--------------+--------------+
|table_name                                         |expectation         |passed_records|failed_records|
+---------------------------------------------------+--------------------+--------------+--------------+
|alexander_booth.pk_unique_demo_dlt.customers_gold  |unique_customer_id  |NULL          |1             |
|alexander_booth.pk_unique_demo_dlt.customers_gold  |unique_email        |NULL          |1             |
|alexander_booth.pk_unique_demo_dlt.customers_silver|customer_id_not_null|38            |0             |
|alexander_booth.pk_unique_demo_dlt.customers_silver|email_not_null      |38            |0             |
|alexander_booth.pk_unique_demo_dlt.customers_silver|first_name_not_null |38            |0             |
|alexander_booth.pk_unique_demo_dlt.customers_silver|last_name_not_null  |38            |0             |
|alexander_booth.

## The Full Picture: RDBMS Constraints → LDP Expectations

| What you had in Oracle/SQL Server | What you use on Databricks |
|-----------------------------------|----------------------------|
| `CHECK (email IS NOT NULL)` | `@dlt.expect_or_drop("email_not_null", "email IS NOT NULL")` |
| `PRIMARY KEY (customer_id)` | `@dlt.expect_or_fail("unique_customer_id", "id_count = 1")` at Gold |
| `UNIQUE (email)` | `@dlt.expect_or_fail("unique_email", "email_count = 1")` at Gold |
| `FOREIGN KEY ... REFERENCES` | `@dlt.expect_or_fail("valid_fk", "parent_exists = true")` at Gold |
| BEFORE INSERT trigger | `@dlt.expect_or_drop` at Silver (filter before it reaches Gold) |
| AFTER INSERT trigger / audit | `@dlt.expect` at any layer (log metric, keep the row for review) |
| Stored procedure with IF/THEN | The pipeline definition itself — declarative, not procedural |

### Why this is actually better than the RDBMS approach

1. **Centralized** — rules are declared once in the pipeline, not scattered across procs/triggers/app code
2. **Observable** — LDP tracks expectation metrics over time (% rows passing, failing, dropped) in the pipeline UI
3. **Graduated** — you choose warn/drop/fail per rule, per layer. In Oracle it's all-or-nothing reject
4. **Auditable** — Bronze keeps the raw data including violations, so you can always go back and investigate
5. **No bypass** — if all writes go through the LDP pipeline, there's no "someone did a raw INSERT" problem

### The recommendation

- **Day 1 (now):** Use `MERGE` + post-write validation in your existing PySpark/SQL jobs (notebook 04)
- **Day 2 (migration):** Move ingestion pipelines to LDP with `expect_or_drop` at Silver, `expect_or_fail` at Gold
- **Always:** Declare PK/UNIQUE/FK constraints in your DDL for optimizer hints and BI tool compatibility, even though they're not enforced

## Cleanup

In [8]:
# Delete the LDP pipeline
w.pipelines.delete(pipeline_id=pipeline_id)
print(f"Deleted pipeline: {pipeline_id}")

# Clean up the backup table
spark.sql(f"DROP TABLE IF EXISTS {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch_full")

# Clean up the LDP target schema
spark.sql(f"DROP SCHEMA IF EXISTS {UC_CATALOG}.{UC_SCHEMA}_dlt CASCADE")

# Remove uploaded workspace file
try:
    w.workspace.delete(path=WORKSPACE_PATH, recursive=True)
    print(f"Deleted workspace folder: {WORKSPACE_PATH}")
except Exception as e:
    print(f"Note: could not delete workspace folder: {e}")

print("\nCleanup complete!")

Deleted pipeline: 7226a463-f784-4a2d-a27a-0a2bd5d73fcf
Deleted workspace folder: /Users/alexander.booth@databricks.com/pk_unique_demo

Cleanup complete!
